# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmdltf/flyrank-intern/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Task Type: Scoring / Regression (Expected CTR Modeling)*

*Why:
This lane is designed as a scoring (or regression) task to predict the Expected CTR (or Opportunity Score) for each page-keyword pair. This continuous score value is then used to rank (rank) the optimization priority queue based on the difference (gap) between actual CTR and potential CTR.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Check target type and initial distribution
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/ahmdltf/flyrank-intern/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

print("Task type: Regression/Scoring on continuous CTR values")
print(f"Target CTR - Min: {df['ctr'].min():.4f}, Max: {df['ctr'].max():.4f}, Mean: {df['ctr'].mean():.4f}")

Task type: Regression/Scoring on continuous CTR values
Target CTR - Min: 0.0000, Max: 100.0000, Mean: 0.5107


## 2. Target or proxy

*Target: opportunity_score (Proxy)*

*Label Origin: This label is a defined rule / proxy metric which is calculated from the results of observations (observed outcome). Since we do not have ground-truth labels directly from humans, we define the opportunity score as the product of the CTR Gap (Expected CTR Position - Actual CTR) by $\log(\text{Impressions})$.*

*$$\text{Opportunity Score} = \max(0, \text{Expected CTR}_{\text{position}} - \text{CTR}_{\text{actual}}) \times \log(1 + \text{Impressions})$$*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Calculates the average CTR benchmark per position as a proxy for expected CTR
position_ctr_benchmark = df.groupby(np.round(df['avg_position']))['ctr'].mean().to_dict()

# Calculate Opportunity Score as a proxy label
df['expected_ctr'] = np.round(df['avg_position']).map(position_ctr_benchmark)
df['ctr_gap'] = (df['expected_ctr'] - df['ctr']).clip(lower=0)
df['opportunity_score'] = df['ctr_gap'] * np.log1p(df['impressions_90d'])

# Automatic detection of available URL/Page column names
page_col = 'url_hash' if 'url_hash' in df.columns else ('url' if 'url' in df.columns else df.columns[0])

print("5 Examples of Pages with the Highest Opportunity Score:")
df[[page_col, 'avg_position', 'impressions_90d', 'ctr', 'opportunity_score']].sort_values(by='opportunity_score', ascending=False).head()

5 Examples of Pages with the Highest Opportunity Score:


,content_id,avg_position,impressions_90d,ctr,opportunity_score
14090,content_44e481c8f55b,1.4,312694,0.65,76.163844
7122,content_7a6df559322d,0.7,43650,0.14,69.760389
27107,content_0022a6b4290f,1.2,29747,0.07,67.977619
7860,content_d225ec9f3d46,0.7,26470,0.05,67.411061
28193,content_6973328a6bb8,1.0,16512,0.07,64.093098


## 3. Success metric

*Metric: Spearman's Rank Correlation ($\rho$) and Mean Absolute Error (MAE) in the Top-20.*

*What number means 'good':*
*   *Spearman $\rho \ge 0.70$ between the model's predicted score ranking and the actual opportunity ranking. This guarantees that the priority order produced by the model is consistent with real conditions in the field.*
*   *MAE $\le 0.05$ in predicting potential CTR on pages at striking distance (positions 4–20).*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error

# Simulated evaluation metrics using a proxy baseline
correlation, _ = spearmanr(df['opportunity_score'], df['impressions_90d'])
mae = mean_absolute_error(df['ctr'], df['expected_ctr'].fillna(0))

print(f"Spearman Basic Correlation: {correlation:.4f}")
print(f"Basic MAE: {mae:.4f}")
print(f"ML Success Target: Spearman >= 0.70 & MAE <= 0.05")

Spearman Basic Correlation: 0.2174
Basic MAE: 0.7025
ML Success Target: Spearman >= 0.70 & MAE <= 0.05


## 4. The unit of analysis, as a real dataframe

*Unit of Analysis: One row = One page (URL/Page) within a 90 day period.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Displays actual analytical units
page_col = 'content_id' if 'content_id' in df.columns else ('url_hash' if 'url_hash' in df.columns else df.columns[0])

slice_df = df[[page_col, 'avg_position', 'impressions_90d', 'clicks_90d', 'ctr']].copy()

print(f"Slice Dataset Shape: {slice_df.shape[0]} row x {slice_df.shape[1]} column")
print(f"1 row = 1 unique unit ({page_col}):")
slice_df.head()

Slice Dataset Shape: 30000 row x 5 column
1 row = 1 unique unit (content_id):


,content_id,avg_position,impressions_90d,clicks_90d,ctr
0,content_304f48230142,10.6,3803,29,0.76
1,content_a1fb4e703a9e,20.3,15320,7,0.05
2,content_9aa793d4d895,36.5,12581,11,0.09
3,content_331d6c4de07b,6.2,11751,58,0.49
4,content_d99b7a2d90ca,44.0,19140,24,0.13


## 5. Why ML beats a fixed rule here

*Why ML beats an if-statement:*

*Aturan tetap (if-statement) seperti IF position <= 10 AND impressions > 1000 THEN prioritize sangat rapuh karena hubungan antara Posisi, CTR, Intent, dan Volume bersifat non-linear dan saling berinteraksi secara kompleks.*

*ML dapat mempelajari kurva CTR decay non-linear spesifik tiap kategori halaman dan menyerap hubungan dinamis antar variabel tanpa terpatok pada ambang batas (threshold) kaku yang membuang banyak pola tersembunyi.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Proving non-linear patterns of CTR against Position that cannot be captured with a simple if-statement
sample_positions = [1, 2, 3, 5, 10, 15, 20]
for pos in sample_positions:
    subset = df[np.round(df['avg_position']) == pos]
    print(f"Position {pos:2d} -> CTR Standard Deviation: {subset['ctr'].std():.4f} (High variance within the same position)")

Position  1 -> CTR Standard Deviation: 18.5275 (High variance within the same position)
Position  2 -> CTR Standard Deviation: 9.8496 (High variance within the same position)
Position  3 -> CTR Standard Deviation: 4.8165 (High variance within the same position)
Position  5 -> CTR Standard Deviation: 2.3904 (High variance within the same position)
Position 10 -> CTR Standard Deviation: 1.1292 (High variance within the same position)
Position 15 -> CTR Standard Deviation: 1.7320 (High variance within the same position)
Position 20 -> CTR Standard Deviation: 0.6112 (High variance within the same position)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.